<a href="https://colab.research.google.com/github/Elazar-segal/colab/blob/main/%D7%9E%D7%93%D7%A8%D7%99%D7%9A_%D7%9B%D7%95%D7%A9%D7%A8_claude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mediapipe opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.8 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [1]:
!wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

In [2]:
# ============================================================
# Smart Fitness Trainer - Rep Counter
# MediaPipe Tasks API (v0.10+) - Final Version
# Google Colab with video file input
# ============================================================
#
# HOW TO USE - 3 steps in Colab:
#
# CELL 1 (run once):
#   !pip install mediapipe opencv-python-headless
#   Then: Runtime -> Restart runtime
#
# CELL 2 (run after restart):
#   !wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task
#
# CELL 3 (this file - paste and run):
#   upload_and_run()  <-- at the bottom
# ============================================================

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import math
import os
from google.colab import files


# ============================================================
# POSE CONNECTIONS
# Defined manually - no legacy mediapipe.framework needed
# Each tuple = two landmark indices to connect with a line
# ============================================================
POSE_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),
    (0,4),(4,5),(5,6),(6,8),
    (9,10),
    (11,12),(11,13),(13,15),(15,17),(15,19),(17,19),
    (12,14),(14,16),(16,18),(16,20),(18,20),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(27,29),(27,31),(29,31),
    (24,26),(26,28),(28,30),(28,32),(30,32),
]


# ============================================================
# EXERCISE CONFIGURATION
# landmarks : 3 joint indices forming the measured angle
# up_angle  : angle at REST position (limb straight)
# down_angle: angle at PEAK EFFORT  (limb bent)
#
# NOTE: down_angle values are tuned based on real video testing.
# If reps are not counted, run debug_video() to find your
# actual angle range and adjust accordingly.
# ============================================================
EXERCISES = {
    'Push-Up': {
        'landmarks':   [12, 14, 16],   # shoulder -> elbow -> wrist
        'up_angle':    160,
        'down_angle':   90,
        'description': 'Elbow angle | Up > 160 deg | Down < 90 deg'
    },
    'Squat': {
        'landmarks':   [24, 26, 28],   # hip -> knee -> ankle
        'up_angle':    155,            # tuned from real video (was 160)
        'down_angle':  120,            # tuned from real video (was 90)
        'description': 'Knee angle | Up > 155 deg | Down < 120 deg'
    },
    'Bicep Curl': {
        'landmarks':   [12, 14, 16],   # shoulder -> elbow -> wrist
        'up_angle':    160,
        'down_angle':   50,
        'description': 'Elbow angle | Up > 160 deg | Down < 50 deg'
    },
}

# MediaPipe landmark index reference:
#   11=left shoulder   12=right shoulder
#   13=left elbow      14=right elbow
#   15=left wrist      16=right wrist
#   23=left hip        24=right hip
#   25=left knee       26=right knee
#   27=left ankle      28=right ankle


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def calculate_angle(a, b, c):
    """
    Calculate the angle in degrees at joint b,
    given three 2D points: a (top), b (joint), c (bottom).
    Uses atan2 for a robust, quadrant-aware calculation.
    """
    radians = math.atan2(c[1] - b[1], c[0] - b[0]) - \
              math.atan2(a[1] - b[1], a[0] - b[0])
    angle = abs(radians * 180.0 / math.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle


def get_landmark_px(landmark, w, h):
    """Convert normalized landmark (0.0-1.0) to pixel coordinates."""
    return [int(landmark.x * w), int(landmark.y * h)]


def draw_skeleton(frame, pose_landmarks, w, h):
    """
    Draw body skeleton using pure cv2.
    No mediapipe drawing utils needed.
    """
    points = [get_landmark_px(lm, w, h) for lm in pose_landmarks]
    for start_idx, end_idx in POSE_CONNECTIONS:
        if start_idx < len(points) and end_idx < len(points):
            cv2.line(frame,
                     tuple(points[start_idx]),
                     tuple(points[end_idx]),
                     (0, 255, 0), 2)
    for pt in points:
        cv2.circle(frame, tuple(pt), 4, (255, 255, 255), -1)


def draw_dashboard(frame, counter, stage, angle, exercise):
    """
    Semi-transparent info panel on top-left corner.
    Shows: exercise name, rep count, movement stage, joint angle.
    """
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (280, 160), (40, 40, 40), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.putText(frame, 'Exercise: ' + exercise,
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (200, 200, 200), 1, cv2.LINE_AA)
    cv2.putText(frame, 'REPS',
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (200, 200, 200), 1, cv2.LINE_AA)
    cv2.putText(frame, str(counter),
                (10, 120), cv2.FONT_HERSHEY_SIMPLEX,
                2.5, (0, 255, 0), 4, cv2.LINE_AA)
    stage_color = (0, 200, 255) if stage == 'UP' else (0, 100, 255)
    cv2.putText(frame, 'Stage: ' + (stage if stage else '---'),
                (10, 145), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, stage_color, 2, cv2.LINE_AA)
    cv2.putText(frame, 'Angle: ' + str(int(angle)),
                (160, 145), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (255, 255, 100), 1, cv2.LINE_AA)


# ============================================================
# DEBUG FUNCTION
# Run this first on a new video to find the correct
# angle thresholds before processing the full video.
# ============================================================

def debug_video(video_path, exercise_name='Squat',
                model_path='pose_landmarker.task'):
    """
    Samples every 10th frame and prints the joint angle.
    Use this to check if up_angle / down_angle need tuning.
    At the end prints the min/max angles seen and suggests fixes.
    """
    if not os.path.exists(model_path):
        print('[ERROR] Model file not found. Run CELL 2 first.')
        return
    if exercise_name not in EXERCISES:
        print('[ERROR] Unknown exercise: ' + exercise_name)
        return

    config        = EXERCISES[exercise_name]
    lm_a, lm_b, lm_c = config['landmarks']
    UP_THRESH     = config['up_angle']
    DOWN_THRESH   = config['down_angle']

    cap = cv2.VideoCapture(video_path)
    base_options = mp_python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )
    landmarker = vision.PoseLandmarker.create_from_options(options)

    frame_count = 0
    min_angle   = 999
    max_angle   = 0

    print('Exercise : ' + exercise_name)
    print('Joints   : landmarks ' + str(lm_a) + ' -> ' + str(lm_b) + ' -> ' + str(lm_c))
    print('UP   threshold : angle > ' + str(UP_THRESH))
    print('DOWN threshold : angle < ' + str(DOWN_THRESH))
    print('-' * 55)
    print('Frame  | Detected |   Angle | Stage')
    print('-' * 55)

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        frame_count += 1
        if frame_count % 10 != 0:
            continue

        rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_img)

        if result.pose_landmarks:
            lms     = result.pose_landmarks[0]
            w, h    = frame.shape[1], frame.shape[0]
            point_a = get_landmark_px(lms[lm_a], w, h)
            point_b = get_landmark_px(lms[lm_b], w, h)
            point_c = get_landmark_px(lms[lm_c], w, h)
            angle   = calculate_angle(point_a, point_b, point_c)
            min_angle = min(min_angle, angle)
            max_angle = max(max_angle, angle)
            if angle > UP_THRESH:
                stage = 'UP   -- rest'
            elif angle < DOWN_THRESH:
                stage = 'DOWN -- peak <<<'
            else:
                stage = 'between (' + str(int(angle)) + ' deg)'
            print(str(frame_count).rjust(6) + ' |      YES | ' +
                  str(round(angle, 1)).rjust(7) + ' | ' + stage)
        else:
            print(str(frame_count).rjust(6) + ' |       NO |     --- | no person detected')

    cap.release()
    landmarker.close()
    print('-' * 55)
    print('Angle range seen : ' + str(int(min_angle)) + ' deg  to  ' + str(int(max_angle)) + ' deg')
    print('For a rep to count, need: > ' + str(UP_THRESH) + ' AND < ' + str(DOWN_THRESH))
    if max_angle < UP_THRESH:
        print('>> PROBLEM: angle never reached UP threshold!')
        print('   Fix: lower up_angle to ' + str(int(max_angle) - 5))
    if min_angle > DOWN_THRESH:
        print('>> PROBLEM: angle never reached DOWN threshold!')
        print('   Fix: raise down_angle to ' + str(int(min_angle) + 5))
    if max_angle >= UP_THRESH and min_angle <= DOWN_THRESH:
        print('>> Thresholds OK -- reps should be counted correctly')


# ============================================================
# MAIN PROCESSING FUNCTION
# ============================================================

def process_video(video_path, exercise_name='Squat',
                  output_path='output.mp4',
                  model_path='pose_landmarker.task'):
    """
    Frame-by-frame video analysis with State Machine rep counting.

    State Machine logic:
        angle > up_angle   --> stage = UP   (rest position)
        angle < down_angle --> stage = DOWN  (peak effort)
        DOWN -> UP          --> counter += 1  (rep completed)

    Tip: run debug_video() first on a new video to verify
    that up_angle and down_angle values are correct.
    """
    if not os.path.exists(model_path):
        print('[ERROR] Model file not found: ' + model_path)
        print('[INFO]  Run CELL 2 to download it.')
        return None
    if exercise_name not in EXERCISES:
        print('[ERROR] Unknown exercise: ' + exercise_name)
        print('        Available: ' + str(list(EXERCISES.keys())))
        return None

    config        = EXERCISES[exercise_name]
    lm_a, lm_b, lm_c = config['landmarks']
    UP_THRESH     = config['up_angle']
    DOWN_THRESH   = config['down_angle']

    print('[INFO] Exercise     : ' + exercise_name)
    print('[INFO] Configuration: ' + config['description'])
    print('[INFO] Opening video: ' + video_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print('[ERROR] Cannot open video: ' + video_path)
        return None

    fps          = cap.get(cv2.CAP_PROP_FPS) or 25
    frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print('[INFO] Resolution   : ' + str(frame_width) + 'x' + str(frame_height) +
          '  FPS: ' + str(round(fps, 1)) +
          '  Frames: ' + str(total_frames))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    base_options = mp_python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        output_segmentation_masks=False,
        min_pose_detection_confidence=0.6,
        min_pose_presence_confidence=0.6,
        min_tracking_confidence=0.6
    )
    landmarker = vision.PoseLandmarker.create_from_options(options)

    counter       = 0
    stage         = None
    current_angle = 0.0
    frame_count   = 0
    first_frame   = True      # used to initialize stage on first detected frame

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        frame_count += 1

        # Resize to max 640px wide before detection
        # Handles 4K and portrait videos - keeps aspect ratio
        # Does not affect angle calculation accuracy
        scale  = 640 / frame.shape[1]
        small  = cv2.resize(frame, (640, int(frame.shape[0] * scale)))
        rgb    = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_img)

        if result.pose_landmarks:
            landmarks = result.pose_landmarks[0]
            # Use small frame dimensions for landmark coordinate calculation
            s_w = small.shape[1]
            s_h = small.shape[0]
            point_a = get_landmark_px(landmarks[lm_a], s_w, s_h)
            point_b = get_landmark_px(landmarks[lm_b], s_w, s_h)
            point_c = get_landmark_px(landmarks[lm_c], s_w, s_h)
            current_angle = calculate_angle(point_a, point_b, point_c)

            # ---------- STATE MACHINE ----------
            # Save previous stage BEFORE any updates
            prev_stage = stage

            # Initialize stage on very first detected frame
            if first_frame:
                if current_angle > UP_THRESH:
                    stage = 'UP'
                else:
                    stage = 'DOWN'   # video started mid-movement
                first_frame = False
                prev_stage  = stage  # no rep on first frame

            # Update stage based on current angle
            if current_angle > UP_THRESH:
                stage = 'UP'
            elif current_angle < DOWN_THRESH:
                stage = 'DOWN'

            # Count rep: only when transitioning FROM DOWN TO UP
            # Uses prev_stage so the transition check is not
            # overwritten by the stage update above
            if prev_stage == 'DOWN' and stage == 'UP':
                counter += 1
                print('    [REP] #' + str(counter) +
                      ' at frame ' + str(frame_count) +
                      '  (angle: ' + str(int(current_angle)) + ' deg)')
            # -----------------------------------

            # Draw on small frame, then scale back to original size
            draw_skeleton(small, landmarks, s_w, s_h)
            for pt, color in zip(
                [point_a, point_b, point_c],
                [(255, 255, 0), (0, 255, 255), (255, 255, 0)]
            ):
                cv2.circle(small, tuple(pt), 8, color, -1)
            cv2.putText(small, str(int(current_angle)) + ' deg',
                        (point_b[0] - 40, point_b[1] - 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2, cv2.LINE_AA)
            # Scale annotated small frame back to original output size
            frame = cv2.resize(small, (frame_width, frame_height))
        else:
            cv2.putText(frame, 'No person detected',
                        (frame_width // 2 - 140, frame_height // 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2, cv2.LINE_AA)

        draw_dashboard(frame, counter, stage, current_angle, exercise_name)

        if frame_count % 30 == 0:
            pct = int(frame_count / total_frames * 100) if total_frames > 0 else 0
            print('[INFO] Progress: ' + str(pct) + '% (' +
                  str(frame_count) + '/' + str(total_frames) + ')', end='\r')
        out.write(frame)

    cap.release()
    out.release()
    landmarker.close()
    print('')
    print('[DONE] Total reps : ' + str(counter))
    print('[DONE] Output     : ' + output_path)
    return counter


# ============================================================
# UPLOAD AND RUN
# ============================================================

def upload_and_run():
    """
    Interactive Colab entry point.
    Tip: if reps are not counted, cancel and run debug_video() first.
    """
    print('=' * 55)
    print('     Smart Fitness Trainer -- Rep Counter')
    print('=' * 55)

    model_path = 'pose_landmarker.task'
    if not os.path.exists(model_path):
        print('[ERROR] Model file not found!')
        print('[INFO]  Run CELL 2 first to download the model.')
        return
    print('[INFO] Model found: ' + model_path)

    print('')
    print('Available exercises:')
    exercise_list = list(EXERCISES.keys())
    for i, name in enumerate(exercise_list, 1):
        cfg = EXERCISES[name]
        print('  ' + str(i) + '. ' + name.ljust(15) +
              '  (' + cfg['description'] + ')')

    choice = input('\nEnter exercise number (default = 1): ').strip()
    try:
        exercise_name = exercise_list[int(choice) - 1]
    except (ValueError, IndexError):
        exercise_name = exercise_list[0]
        print('[INFO] Defaulting to: ' + exercise_name)

    print('[INFO] Selected   : ' + exercise_name)
    print('[INFO] Upload your video file (mp4, mov, avi)...')

    uploaded = files.upload()
    if not uploaded:
        print('[ERROR] No file uploaded.')
        return

    video_filename  = list(uploaded.keys())[0]
    output_filename = 'output_annotated.mp4'

    print('[INFO] File received: ' + video_filename)
    print('[INFO] Tip: if reps = 0, cancel and run debug_video() to tune thresholds')
    print('')

    total_reps = process_video(
        video_path=video_filename,
        exercise_name=exercise_name,
        output_path=output_filename,
        model_path=model_path
    )

    if total_reps is not None:
        print('')
        print('=' * 55)
        print('  FINAL RESULT: ' + str(total_reps) + ' reps of ' + exercise_name)
        print('=' * 55)
        print('[INFO] Downloading annotated video...')
        files.download(output_filename)
        print('[INFO] Done! Check your Downloads folder.')


# ============================================================
# ENTRY POINT
# ============================================================
#
# Option A - full interactive run:
upload_and_run()
#
# Option B - debug first (paste video filename):
# debug_video('your_video.mp4', exercise_name='Squat')
#
# Option C - process directly without upload prompt:
# process_video('your_video.mp4', exercise_name='Squat')

     Smart Fitness Trainer -- Rep Counter
[INFO] Model found: pose_landmarker.task

Available exercises:
  1. Push-Up          (Elbow angle | Up > 160 deg | Down < 90 deg)
  2. Squat            (Knee angle | Up > 155 deg | Down < 120 deg)
  3. Bicep Curl       (Elbow angle | Up > 160 deg | Down < 50 deg)

Enter exercise number (default = 1): 2
[INFO] Selected   : Squat
[INFO] Upload your video file (mp4, mov, avi)...


Saving 4838220-uhd_2160_3840_24fps.mp4 to 4838220-uhd_2160_3840_24fps (2).mp4
[INFO] File received: 4838220-uhd_2160_3840_24fps (2).mp4
[INFO] Tip: if reps = 0, cancel and run debug_video() to tune thresholds

[INFO] Exercise     : Squat
[INFO] Configuration: Knee angle | Up > 155 deg | Down < 120 deg
[INFO] Opening video: 4838220-uhd_2160_3840_24fps (2).mp4
[INFO] Resolution   : 2160x3840  FPS: 24.0  Frames: 304
    [REP] #1 at frame 72  (angle: 156 deg)
    [REP] #2 at frame 187  (angle: 156 deg)

[DONE] Total reps : 2
[DONE] Output     : output_annotated.mp4

  FINAL RESULT: 2 reps of Squat
[INFO] Downloading annotated video...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[INFO] Done! Check your Downloads folder.
